# COVID Data Analysis

This notebook follows the tasks defined in `plan.md` and uses only the three products named there.


## 1. Read plan.md


In [1]:
from pathlib import Path
import re

workspace = Path(r"c:\Users\juanc\GitHub\LAB-MAA")
if not workspace.exists():
    workspace = Path.cwd()

plan_path = workspace / "plan.md"
covid_dir = workspace / "data" / "COVID"
output_dir = workspace / "outputs" / "COVID"
output_dir.mkdir(parents=True, exist_ok=True)

plan_text = plan_path.read_text(encoding="utf-8")
plan_lines = [line.strip() for line in plan_text.splitlines() if line.strip()]
task_lines = [
    line for line in plan_lines
    if line.startswith(("-", "*", "+"))
    or re.match(r"^\d+[\.\)]", line)
    or line.startswith("#")
]

print(f"Plan: {plan_path}")
print(f"Identified {len(task_lines)} task or heading lines")
print("\n".join(task_lines))

Plan: c:\Users\juanc\GitHub\LAB-MAA\plan.md
Identified 24 task or heading lines
# Trabajo Machine Learning — Vacunación COVID-19 y Mortalidad por Zona (Chile)
- y_zona,t: mortalidad en la zona "zona en el período" t, medida como
- X_zona,t: porcentaje de población vacunada (al menos 1ª dosis, o esquema
- b_zona: coeficiente específico por zona (o interacción Zona × X), para
1. Unidad temporal común: se agregan ambas series a frecuencia semanal
2. La tasa de vacunación es exógena respecto a la mortalidad contemporánea
3. Independencia entre zonas
4. Los fallecidos y la vacunación están correctamente asignados a la comuna/
5. Zonificación usada:
- Norte: Arica y Parinacota, Tarapacá, Antofagasta, Atacama, Coquimbo
- Centro: Valparaíso, Metropolitana, O'Higgins, Maule
- Sur: Ñuble, Biobío, La Araucanía, Los Ríos, Los Lagos, Aysén, Magallanes
- data/COVID/datos-covid-19/producto84/fallecidos_comuna_edad_totales_std.csv | Region, Fecha, Total |
- data/COVID/datos-covid-19/producto83/vacunac

## 2. Inspect data/COVID


In [2]:
from pathlib import Path
import pandas as pd

workspace = Path(r"c:\Users\juanc\GitHub\LAB-MAA")
if not workspace.exists():
    workspace = Path.cwd()
covid_dir = workspace / "data" / "COVID"
output_dir = workspace / "outputs" / "COVID"
output_dir.mkdir(parents=True, exist_ok=True)

# Use only the three products named in plan.md (no recursive folder scan).
product_dir = covid_dir / "datos-covid-19"
source_paths = {
    "deaths":      product_dir / "producto84" / "fallecidos_comuna_edad_totales_std.csv",
    "vaccination": product_dir / "producto83" / "vacunacion_establecimiento_std.csv",
    "population":  product_dir / "producto93" / "ContactosPorComuna.csv",
}
missing_sources = [path for path in source_paths.values() if not path.exists()]
if missing_sources:
    raise FileNotFoundError(f"Missing planned source(s): {missing_sources}")

data_catalog = pd.DataFrame([
    {"product": product, "file": str(path.relative_to(workspace)), "exists": path.exists()}
    for product, path in source_paths.items()
])
data_catalog.to_csv(output_dir / "data_catalog.csv", index=False)
display(data_catalog)

,product,file,exists
0,deaths,data\COVID\datos-covid-19\producto84\fallecido...,True
1,vaccination,data\COVID\datos-covid-19\producto83\vacunacio...,True
2,population,data\COVID\datos-covid-19\producto93\Contactos...,True


In [3]:
# Load only the columns used by the planned ETL.
deaths_raw = pd.read_csv(
    source_paths["deaths"],
    usecols=["Region", "Fecha", "Total"],
)
vaccination_raw = pd.read_csv(
    source_paths["vaccination"],
    usecols=["Establecimiento", "Fecha", "Dosis", "Cantidad"],
)
population_raw = pd.read_csv(
    source_paths["population"],
    usecols=["Region", "Codigo comuna", "Comuna", "Poblacion"],
)

datasets = {
    "producto84_fallecidos": deaths_raw,
    "producto83_vacunacion": vaccination_raw,
    "producto93_poblacion": population_raw,
}
quality_df = pd.DataFrame([
    {
        "product": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in datasets.items()
])
quality_df.to_csv(output_dir / "data_quality_report.csv", index=False)
display(quality_df)
print({name: frame.shape for name, frame in datasets.items()})

,product,rows,columns,duplicate_rows,missing_cells
0,producto84_fallecidos,1940022,3,1916121,0
1,producto83_vacunacion,1449882,4,0,0
2,producto93_poblacion,346,4,0,0


{'producto84_fallecidos': (1940022, 3), 'producto83_vacunacion': (1449882, 4), 'producto93_poblacion': (346, 4)}


## 3. Implement the planned analysis


In [4]:
import re
import unicodedata
import numpy as np
import pandas as pd

# --- Zone lookup (plan): keywords per zone, robust to accents / official names ---
zone_aliases = {
    "Norte": ["arica y parinacota", "tarapaca", "antofagasta", "atacama", "coquimbo"],
    "Centro": ["valparaiso", "metropolitana", "ohiggins", "maule"],
    "Sur":   ["nuble", "biobio", "la araucania", "los rios", "los lagos", "aysen", "magallanes"],
}
keyword_zone = {kw: zone for zone, kws in zone_aliases.items() for kw in kws}
keyword_list = sorted(keyword_zone, key=len, reverse=True)

def clean_text(value):
    value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", value.lower()).strip()

def assign_zone(region):
    # Direct match first, then substring fallback for names that differ from the aliases.
    key = clean_text(region)
    zone = keyword_zone.get(key)
    if zone is None:
        zone = next((keyword_zone[k] for k in keyword_list if k in key), None)
    return zone

# Built once over the observed region names and reused for deaths + population.
region_zone = pd.Series({
    region: assign_zone(region)
    for region in set(deaths_raw["Region"]).union(population_raw["Region"])
})

# 1) Deaths: aggregate Region+Fecha first, collapsing Comuna/Edad (plan step 1).
deaths = deaths_raw.groupby(["Region", "Fecha"], as_index=False, sort=False)["Total"].sum()
deaths = deaths.rename(columns={"Region": "region", "Fecha": "date", "Total": "deaths"})
deaths["date"] = pd.to_datetime(deaths["date"], errors="coerce")
deaths["zone"] = deaths["region"].map(region_zone)
deaths_weekly = (
    deaths.dropna(subset=["date", "zone"])
    .assign(week=lambda f: f["date"].dt.to_period("W-SUN").dt.start_time)
    .groupby(["zone", "week"], as_index=False, sort=False)["deaths"].sum()
)

# 2) Population: one value per commune (plan step 2).
population = population_raw.rename(columns={"Region": "region", "Comuna": "commune", "Poblacion": "population"})
population["population"] = pd.to_numeric(population["population"], errors="coerce")
population["zone"] = population["region"].map(region_zone)
population = population.dropna(subset=["zone", "population"]).drop_duplicates(subset=["Codigo comuna"])
population_zone = population.groupby("zone", as_index=False, sort=False)["population"].sum()
population_region = population.groupby(["region", "zone"], as_index=False, sort=False)["population"].sum()

# 3) Vaccination: keep first doses only when source has several dose labels (plan step 3).
commune_to_zone = {
    clean_text(commune): zone
    for commune, zone in population[["commune", "zone"]].drop_duplicates().itertuples(index=False)
}
commune_keys = sorted((key for key in commune_to_zone if key), key=len, reverse=True)
commune_pattern = re.compile(
    r"(?<![a-z])(" + "|".join(re.escape(key) for key in commune_keys) + r")(?![a-z])"
)

dose_text = vaccination_raw["Dosis"].astype("string").str.lower()
dose_labels = dose_text.dropna().unique()
if len(dose_labels) > 1:
    first_dose = dose_text.str.contains(r"1|prim|unica", regex=True, na=False)
    vaccination = vaccination_raw.loc[first_dose, ["Establecimiento", "Fecha", "Cantidad"]].copy()
else:
    vaccination = vaccination_raw[["Establecimiento", "Fecha", "Cantidad"]].copy()
vaccination = vaccination.rename(columns={"Establecimiento": "establishment", "Fecha": "date", "Cantidad": "vaccinated"})
vaccination["vaccinated"] = pd.to_numeric(vaccination["vaccinated"], errors="coerce").fillna(0)

# 4) Assign each unique establishment to a zone via the compiled commune regex (plan step 4).
establishment_zone_map = {}
for establishment in vaccination["establishment"].dropna().unique():
    match = commune_pattern.search(clean_text(establishment))
    establishment_zone_map[establishment] = commune_to_zone[match.group(1)] if match else pd.NA
vaccination["zone"] = vaccination["establishment"].map(establishment_zone_map)
vaccination_unmapped = vaccination.loc[vaccination["zone"].isna(), "establishment"].drop_duplicates()

vaccination_weekly = (
    vaccination.dropna(subset=["date", "zone"])
    .assign(date=lambda f: pd.to_datetime(f["date"], errors="coerce"))
    .dropna(subset=["date"])
    .assign(week=lambda f: f["date"].dt.to_period("W-SUN").dt.start_time)
    .groupby(["zone", "week"], as_index=False, sort=False)["vaccinated"].sum()
)
vaccination_weekly = vaccination_weekly.sort_values(["zone", "week"])
vaccination_weekly["vaccinated_accumulated"] = vaccination_weekly.groupby("zone")["vaccinated"].cumsum()

# 5-6) Merge into a zone x week panel and compute rates (plan steps 5-6).
panel = deaths_weekly.merge(vaccination_weekly, on=["zone", "week"], how="outer")
panel = panel.merge(population_zone, on="zone", how="left")
panel = panel.sort_values(["zone", "week"]).reset_index(drop=True)
panel[["deaths", "vaccinated", "vaccinated_accumulated"]] = panel[["deaths", "vaccinated", "vaccinated_accumulated"]].fillna(0)
panel["mortality_rate_per_100k"] = panel["deaths"] / panel["population"] * 100000
panel["pct_vaccinated"] = panel["vaccinated_accumulated"] / panel["population"] * 100
panel = panel.dropna(subset=["population"])

panel.to_csv(output_dir / "covid_zone_weekly_panel.csv", index=False)
population_region.to_csv(output_dir / "population_by_region_zone.csv", index=False)
pd.DataFrame({"unmapped_establishments": vaccination_unmapped}).to_csv(
    output_dir / "vaccination_unmapped_establishments.csv", index=False
)
print(f"Panel shape: {panel.shape}")
print(f"Unmapped vaccination establishments: {len(vaccination_unmapped)}")
display(panel.head())

Panel shape: (456, 8)
Unmapped vaccination establishments: 1235


,zone,week,deaths,vaccinated,vaccinated_accumulated,population,mortality_rate_per_100k,pct_vaccinated
0,Centro,2020-03-16,8.0,0.0,0.0,12208244.0,0.065529,0.0
1,Centro,2020-03-23,7.0,0.0,0.0,12208244.0,0.057338,0.0
2,Centro,2020-03-30,32.0,0.0,0.0,12208244.0,0.262118,0.0
3,Centro,2020-04-06,56.0,0.0,0.0,12208244.0,0.458706,0.0
4,Centro,2020-04-13,67.0,0.0,0.0,12208244.0,0.548809,0.0


In [5]:
# Estimate the planned zone-specific linear relationship with OLS interactions.
regression_data = panel.dropna(subset=["mortality_rate_per_100k", "pct_vaccinated"]).copy()
regression_data["intercept"] = 1.0
for zone in ["Norte", "Centro", "Sur"]:
    regression_data[f"vaccination_{zone.lower()}"] = (
        regression_data["pct_vaccinated"] * (regression_data["zone"] == zone)
    )

predictors = ["intercept"] + [f"vaccination_{zone.lower()}" for zone in ["Norte", "Centro", "Sur"]]
design = regression_data[predictors].to_numpy(dtype=float)
response = regression_data["mortality_rate_per_100k"].to_numpy(dtype=float)
coefficients, *_ = np.linalg.lstsq(design, response, rcond=None)
regression_results = pd.DataFrame({"term": predictors, "coefficient": coefficients})
regression_results.to_csv(output_dir / "zone_vaccination_ols.csv", index=False)
display(regression_results)
print(f"Observations used: {len(regression_data)}")

,term,coefficient
0,intercept,2.984130
1,vaccination_norte,-0.022234
2,vaccination_centro,-0.024159
3,vaccination_sur,-0.014988


Observations used: 456


## 4. Validate outputs


In [6]:
import json

required_outputs = [
    output_dir / "data_catalog.csv",
    output_dir / "data_quality_report.csv",
    output_dir / "covid_zone_weekly_panel.csv",
    output_dir / "population_by_region_zone.csv",
    output_dir / "vaccination_unmapped_establishments.csv",
    output_dir / "zone_vaccination_ols.csv",
]

validation = {
    "plan_exists": (workspace / "plan.md").exists(),
    "data_directory_exists": covid_dir.exists(),
    "products_used": list(source_paths),
    "panel_rows": int(len(panel)),
    "panel_zones": sorted(panel["zone"].dropna().unique().tolist()),
    "required_outputs": {
        str(path.relative_to(workspace)): path.exists()
        for path in required_outputs
    },
}
validation["all_required_outputs_exist"] = all(validation["required_outputs"].values())
validation_path = output_dir / "validation_report.json"
validation_path.write_text(json.dumps(validation, indent=2), encoding="utf-8")
print(json.dumps(validation, indent=2))
if not validation["all_required_outputs_exist"]:
    raise RuntimeError("One or more required analysis outputs were not created.")

{
  "plan_exists": true,
  "data_directory_exists": true,
  "products_used": [
    "deaths",
    "vaccination",
    "population"
  ],
  "panel_rows": 456,
  "panel_zones": [
    "Centro",
    "Norte",
    "Sur"
  ],
  "required_outputs": {
    "outputs\\COVID\\data_catalog.csv": true,
    "outputs\\COVID\\data_quality_report.csv": true,
    "outputs\\COVID\\covid_zone_weekly_panel.csv": true,
    "outputs\\COVID\\population_by_region_zone.csv": true,
    "outputs\\COVID\\vaccination_unmapped_establishments.csv": true,
    "outputs\\COVID\\zone_vaccination_ols.csv": true
  },
  "all_required_outputs_exist": true
}
